# 심화 미션: 스마트팜 온실 출하 기록
- 상황: 선별대에서 도장을 찍기 전에 등외를 미리 알고 싶다
- 목표: 오늘 배운 순서를 다른 데이터로 혼자 한 바퀴 돌린다

### 용어 풀이 - 온실에서 쓰는 말

| 말 | 뜻 |
|---|---|
| 상품 / 등외 | 선별대에서 붙이는 판정. 등외는 제값에 못 파는 것 |
| 양액 (EC) | 물에 녹인 거름. 그 진하기를 dS/m 라는 단위로 잰다 |
| 산도 (pH) | 산성인지 알칼리성인지. 7이 중간이고 낮을수록 산성 |
| 토양 수분 | 흙에 물기가 얼마나 있는지 (%) |
| 야간 최저 기온 | 밤에 가장 낮았던 기온. 작물이 스트레스를 받는 지점 |

## Q1. 파일 열고 크기 확인하기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/day03_greenhouse.csv")

print("shape (행, 열):", df.shape)

shape (행, 열): (2000, 13)


## Q2. 등외는 얼마나 드문가

In [2]:
df["result"].value_counts()

result
상품    1861
등외     139
Name: count, dtype: int64

In [3]:
print(df["result"].value_counts(normalize=True) * 100)

result
상품    93.05
등외     6.95
Name: proportion, dtype: float64


## Q3. 정답표를 숫자로 바꾸기

In [4]:
df["등외여부"] = (df["result"] == "등외").astype(int)

print(df["등외여부"].value_counts())

등외여부
0    1861
1     139
Name: count, dtype: int64


## Q4. 입력과 정답으로 가르기

In [5]:
센서열 = ["temp_avg", "humidity_avg", "co2_ppm", "soil_moisture",
        "ec", "ph", "light_hours", "night_temp_min"]

X = df[센서열]
y = df["등외여부"]

print("입력:", X.shape)
print("정답:", y.shape)

입력: (2000, 8)
정답: (2000,)


 ## Q5. 학습용과 시험용으로 나누기

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 시험용으로 떼어둘 비율 (20%)
    random_state=42,      # 무작위로 섞되, 다시 실행해도 같게 나오도록 고정
    stratify=y            # 등외 비율을 양쪽에 똑같이 맞춰서 나눈다
)

print("학습용:", X_train.shape)
print("시험용:", X_test.shape)

print("원본 등외 비율:", round(y.mean() * 100, 2), "%")
print("학습용 등외 비율:", round(y_train.mean() * 100, 2), "%")
print("시험용 등외 비율:", round(y_test.mean() * 100, 2), "%")

학습용: (1600, 8)
시험용: (400, 8)
원본 등외 비율: 6.95 %
학습용 등외 비율: 6.94 %
시험용 등외 비율: 7.0 %


## Q6. 아무것도 배우지 않은 기준 모델

In [7]:
정답맞힌수 = (y_test == 0).sum()      # 실제로 상품인 것 = 다 맞힌 것
print("기준 모델 정확도:", 정답맞힌수 / len(y_test) * 100, "%")

기준 모델 정확도: 93.0 %


## Q7. 진짜 모델 학습시키기

### 모델 추천 - 로지스틱 회귀

- 데이터 상황: 전체 2,000건(학습 1,600 / 시험 400), 입력 숫자열 8개, 등외 비율 약 7%, 열마다 단위가 다름(온도 ℃, CO2 ppm, pH 0~14, 일조시간 등)
- **로지스틱 회귀**를 추천한다
  1. 정답이 상품/등외 둘 중 하나인 **이진분류** 문제라 로지스틱 회귀가 가장 기본이 되는 선택이다
  2. 입력이 전부 숫자형이고 열마다 단위·자릿수가 크게 달라서(예: CO2는 수백 ppm, pH는 0~14) **표준화**만 해주면 바로 쓸 수 있다
  3. 각 센서가 등외 판정에 얼마나, 어느 방향으로 영향을 주는지 계수로 볼 수 있어 **설명하기 쉽다**
  4. 데이터가 2,000건 정도로 크지 않아 복잡한 모델보다 가볍고 안정적인 선형 모델이 과적합 위험이 적다
- 등외가 7%로 드문 편이라 나중에 재현율이 낮게 나올 수 있는데, 그건 다음 단계(가중치 조정 등)에서 다룬다. 지금은 손대지 않은 기본 설정으로 먼저 재본다

In [8]:
# 표준화와 로지스틱 회귀, 채점 도구를 불러온다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

# 열마다 단위가 다르므로, 학습용 기준으로 평균·표준편차를 구해 두 데이터 모두에 적용한다
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 아직 아무 설정도 손대지 않은 기본 상태
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

예측 = model.predict(X_test_scaled)

맞힌상품, 헛경보, 놓친등외, 잡은등외 = confusion_matrix(y_test, 예측).ravel()

print("정확도:", round((예측 == y_test).mean() * 100, 2), "%")
print("잡은 등외:", 잡은등외, "/ 놓친 등외:", 놓친등외, "/ 헛경보:", 헛경보)
print("재현율:", round(recall_score(y_test, 예측), 3),
      "정밀도:", round(precision_score(y_test, 예측, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 예측), 3))

정확도: 94.5 %
잡은 등외: 13 / 놓친 등외: 15 / 헛경보: 7
재현율: 0.464 정밀도: 0.65 F1: 0.542


 ## Q8. 혼동행렬 네 칸 채우기

In [9]:
# confusion_matrix - 실제 답과 예측을 맞대어 네 칸으로 세어준다
from sklearn.metrics import confusion_matrix

행렬 = confusion_matrix(y_test, 예측)
print(행렬)

# ravel() - 네 칸을 한 줄로 펴서 하나씩 이름을 붙여 받는다
# 순서가 정해져 있다: 왼쪽위, 오른쪽위, 왼쪽아래, 오른쪽아래
맞힌양품, 헛경보, 놓친불량, 잡은불량 = 행렬.ravel()

print("양품인데 양품이라 함 :", 맞힌양품)
print("양품인데 불량이라 함 :", 헛경보,   " <- 헛경보")
print("불량인데 양품이라 함 :", 놓친불량, " <- 놓친 등외")
print("불량인데 불량이라 함 :", 잡은불량)

[[365   7]
 [ 15  13]]
양품인데 양품이라 함 : 365
양품인데 불량이라 함 : 7  <- 헛경보
불량인데 양품이라 함 : 15  <- 놓친 등외
불량인데 불량이라 함 : 13


## Q9. 세 가지 지표 구하기

| | 정확도 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|
| 기준 모델 (전부 상품) | | | | |
| 내가 학습시킨 모델 | | | | |

In [10]:
from sklearn.metrics import classification_report

print(classification_report(y_test, 예측,
                            target_names=["상품", "등외"], digits=3))

              precision    recall  f1-score   support

          상품      0.961     0.981     0.971       372
          등외      0.650     0.464     0.542        28

    accuracy                          0.945       400
   macro avg      0.805     0.723     0.756       400
weighted avg      0.939     0.945     0.941       400



## Q10. 놓친 등외를 더 잡으려면

In [11]:
# predict_proba - 각 줄이 등외일 "가능성"을 0~1 숫자로 내놓는다
가능성 = 모델.predict_proba(X_test)[:, 1]

for 문턱 in [0.5, 0.3, 0.2, 0.1]:
    판정 = (가능성 >= 문턱).astype(int)      # 문턱을 넘으면 등외로 본다
    tn, fp, fn, tp = confusion_matrix(y_test, 판정).ravel()
    print(f"문턱 {문턱} | 잡은 등외 {tp:2d} | 헛경보 {fp:2d} | "
          f"재현율 {tp/(tp+fn):.3f} | 정밀도 {tp/(tp+fp):.3f}")

NameError: name '모델' is not defined

In [12]:
# predict_proba - 각 줄이 등외일 "가능성"을 0~1 숫자로 내놓는다
# (수정 1) 모델 변수명은 '모델'이 아니라 'model'
# (수정 2) model은 표준화된 데이터로 학습했으므로 X_test가 아니라 X_test_scaled를 넣는다
가능성 = model.predict_proba(X_test_scaled)[:, 1]

for 문턱 in [0.5, 0.3, 0.2, 0.1]:
    판정 = (가능성 >= 문턱).astype(int)      # 문턱을 넘으면 등외로 본다
    tn, fp, fn, tp = confusion_matrix(y_test, 판정).ravel()
    print(f"문턱 {문턱} | 잡은 등외 {tp:2d} | 헛경보 {fp:2d} | "
          f"재현율 {tp/(tp+fn):.3f} | 정밀도 {tp/(tp+fp):.3f}")

문턱 0.5 | 잡은 등외 13 | 헛경보  7 | 재현율 0.464 | 정밀도 0.650
문턱 0.3 | 잡은 등외 19 | 헛경보 13 | 재현율 0.679 | 정밀도 0.594
문턱 0.2 | 잡은 등외 21 | 헛경보 19 | 재현율 0.750 | 정밀도 0.525
문턱 0.1 | 잡은 등외 25 | 헛경보 36 | 재현율 0.893 | 정밀도 0.410
